In [ ]:
#| default_exp agy

# agy

> Read, search, and compact Antigravity conversations

Find, read, search, and compact Antigravity CLI and desktop conversations stored on this machine.

## Conversation layout

Antigravity CLI stores each conversation as a SQLite database under `AGY_HOME/conversations/<id>.db`, alongside a summary index at `AGY_HOME/conversation_summaries.db` and JSONL transcript mirrors under `AGY_HOME/brain/<id>/.system_generated/logs/`. Desktop conversations use the same schema under `AGY_APP_HOME` (`~/.gemini/antigravity`). Passing `agy_home=None` searches both directories. Use `cur_conv` for a running session's id, `conv_file` to locate a database by id or unique prefix, `project_conv` for the newest conversation in a project, `resolve_conv` to accept an id, prefix, or title, and `prompt_hist` to recover earlier user prompts.

The `steps` table stores ordered protobuf records: user inputs (`14`), planner responses (`15`), tool results (`132`), system messages (`101`), and compaction checkpoints (`23`). `load_steps` decodes every step in the database. `active_steps` starts from the latest checkpoint. `steps2chat` converts steps into canonical messages, unwrapping `call_mcp_tool` calls and preserving untruncated `𝍁…𝍁` outputs.

## Reading workflow

Start with `conv2dlg` to read a conversation as an aidialog dialog. Search with `d.find_msgs(pat)` and read messages with `view_msg`. For step-level inspection, load with `load_conv`, search with `step_search`, and display slices with `show_steps`.

Reading functions open SQLite read-only and never modify databases. Synthetic compaction appends one `CHECKPOINT` row (`step_type = 23`) directly to `steps`. Close Antigravity before running `append_compaction` or `compact_session`.

In [ ]:
#| export
import json, os, re, sqlite3, uuid
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import unquote, urlparse
from fastcore.utils import *
from aidialog.msg_parts import Msg, Text, ToolUse, ToolResult
from aihist.hist import chat2dlg
from fastclaude.session import uniq_path
from llmsurgery.compact import *
from llmsurgery.utils import MAXLEN

In [ ]:
from collections import Counter
from fastcore.test import *
import shutil, tempfile

## Protobuf wire helpers

Antigravity stores step payloads and metadata as standard unencrypted protobuf messages. These wire helpers decode and encode varints and length-delimited fields without adding a protobuf runtime dependency.

In [ ]:
#| export
def _pb_varint(n):
    "Encode unsigned integer `n` as a protobuf base-128 varint"
    out = bytearray()
    while True:
        b = n & 0x7f
        n >>= 7
        if n: out.append(b | 0x80)
        else:
            out.append(b)
            return bytes(out)

def _pb_read_varint(buf, pos):
    res = shift = 0
    while True:
        b = buf[pos]
        pos += 1
        res |= (b & 0x7f) << shift
        if not (b & 0x80): return res, pos
        shift += 7

def _pb_decode(buf):
    "Decode protobuf wire bytes into `(field_num, wire_type, value)` tuples"
    pos, n, out = 0, len(buf), []
    while pos < n:
        tag, pos = _pb_read_varint(buf, pos)
        fn, wt = tag >> 3, tag & 7
        if wt == 0: v, pos = _pb_read_varint(buf, pos)
        elif wt == 1: v, pos = buf[pos:pos+8], pos+8
        elif wt == 2:
            ln, pos = _pb_read_varint(buf, pos)
            v, pos = buf[pos:pos+ln], pos+ln
        elif wt == 5: v, pos = buf[pos:pos+4], pos+4
        else: raise ValueError(f'Unknown protobuf wire type {wt} at offset {pos}')
        out.append((fn, wt, v))
    return out

def _pb_f_int(fn, v): return _pb_varint((fn << 3) | 0) + _pb_varint(v)
def _pb_f_bytes(fn, b):
    if isinstance(b, str): b = b.encode('utf-8')
    return _pb_varint((fn << 3) | 2) + _pb_varint(len(b)) + b

def _pb_one(fields, fn, default=None): return next((v for f,_,v in fields if f == fn), default)
def _pb_all(fields, fn): return [v for f,_,v in fields if f == fn]
def _pb_str(fields, fn, default=''):
    v = _pb_one(fields, fn)
    return v.decode('utf-8', 'replace') if isinstance(v, bytes) else default

In [ ]:
#| export
def _pb_ts(buf):
    "Decode a protobuf Timestamp message into an ISO-8601 UTC string"
    if not buf: return ''
    fs = _pb_decode(buf)
    sec, ns = _pb_one(fs, 1, 0), _pb_one(fs, 2, 0)
    dt = datetime.fromtimestamp(sec + ns/1e9, timezone.utc)
    return dt.isoformat(timespec='milliseconds').replace('+00:00', 'Z')

def _pb_ts_bytes(dt=None):
    "Encode a datetime (default: current UTC time) as a protobuf Timestamp"
    dt = dt or datetime.now(timezone.utc)
    sec = int(dt.timestamp())
    return _pb_f_int(1, sec) + _pb_f_int(2, dt.microsecond * 1000)

Check round-trip encoding for integers, strings, and timestamps:

In [ ]:
ts_dt = datetime(2026, 11, 20, 2, 56, 12, 469000, timezone.utc)
wire = _pb_f_int(1, 23) + _pb_f_bytes(5, 'summary') + _pb_f_bytes(8, _pb_ts_bytes(ts_dt))
dec = _pb_decode(wire)
test_eq(_pb_one(dec, 1), 23)
test_eq(_pb_str(dec, 5), 'summary')
test_eq(_pb_ts(_pb_one(dec, 8)), '2026-11-20T02:56:12.469Z')
dec

## Where conversations live

Antigravity CLI stores databases in `AGY_HOME/conversations` and indexes them in `AGY_HOME/conversation_summaries.db`. Commands started inside Antigravity inherit `ANTIGRAVITY_CONVERSATION_ID`. `conv_file` looks up `<id>.db` directly first, then checks `conversation_summaries.db` so desktop conversations under sibling app data directories resolve by id too.

In [ ]:
#| export
AGY_HOME = Path.home()/'.gemini'/'antigravity-cli'
AGY_APP_HOME = Path.home()/'.gemini'/'antigravity'

STEP_USER, STEP_ASST, STEP_CHECKPOINT, STEP_SYS, STEP_TOOL = 14, 15, 23, 101, 132
STATUS_DONE, STATUS_CANCELED, STATUS_ERROR = 3, 6, 7

def agy_homes(
    agy_home=None, # Single app data directory, list of directories, or None for both CLI and desktop stores
):
    "App data directories to search for Antigravity conversations"
    if agy_home is None: return [AGY_HOME, AGY_APP_HOME]
    if isinstance(agy_home, (str, Path)): return [Path(agy_home)]
    return [Path(h) for h in agy_home]

def _wal_active(path):
    try: return Path(f'{path}-wal').stat().st_size > 0
    except FileNotFoundError: return False

def db_mtime_ns(
    path, # Path to a transcript file or SQLite `.db`
):
    "Modification time in nanoseconds, including a non-empty SQLite `-wal` file when present"
    p = Path(path)
    mt = p.stat().st_mtime_ns
    try: st = Path(f'{p}-wal').stat()
    except FileNotFoundError: return mt
    return max(mt, st.st_mtime_ns) if st.st_size > 0 else mt

def cur_conv():
    "The current Antigravity conversation id, when running inside Antigravity"
    return os.environ.get('ANTIGRAVITY_CONVERSATION_ID')

def _ro_conn(path):
    q = 'mode=ro' if _wal_active(path) else 'mode=ro&immutable=1'
    return sqlite3.connect(f'{Path(path).resolve().as_uri()}?{q}', uri=True)

def _summary_rows(agy_home=None):
    rows = []
    for h in agy_homes(agy_home):
        p = h/'conversation_summaries.db'
        if not p.exists(): continue
        with _ro_conn(p) as con:
            sql = ('SELECT conversation_id, title, workspace_uris, last_modified_time, parent_conversation_id, app_data_dir '
                'FROM conversation_summaries ORDER BY last_modified_time DESC')
            for r in con.execute(sql).fetchall():
                rows.append(AttrDict(cid=r[0], title=r[1], workspace_uris=r[2], last_modified=r[3], parent=r[4], app_dir=r[5], home=h))
    return sorted(rows, key=lambda r: r.last_modified, reverse=True)

def _db_for_summary(row):
    p = row.home/'conversations'/f'{row.cid}.db'
    if p.exists(): return p
    if row.app_dir:
        sib = row.home.parent/row.app_dir/'conversations'/f'{row.cid}.db'
        if sib.exists(): return sib
    return None

In [ ]:
#| export
def conv_file(
    conv_id, # Conversation UUID, unique id prefix, or path to a `.db` file
    agy_home=None, # Antigravity app data directory, list of directories, or None for CLI and desktop
):
    "The conversation database for `conv_id`, if it exists"
    if not conv_id: return None
    if isinstance(conv_id, Path) or '/' in conv_id:
        p = Path(conv_id)
        return p if p.is_file() else None
    found = {}
    for h in agy_homes(agy_home):
        cdir = h/'conversations'
        if cdir.exists():
            for p in sorted(cdir.glob(f'{conv_id}*.db')): found.setdefault(p.stem, p)
    if not found:
        for r in _summary_rows(agy_home):
            if r.cid.startswith(conv_id) and (p := _db_for_summary(r)): found.setdefault(r.cid, p)
    return uniq_path(found.values(), conv_id)

def _db_info(path):
    with _ro_conn(path) as con: row = con.execute("SELECT data FROM trajectory_metadata_blob WHERE id='main'").fetchone()
    if not row or not row[0]: return '', ''
    fs = _pb_decode(row[0])
    uri = _pb_str(fs, 7) or (_pb_str(_pb_decode(ws), 1) if (ws := _pb_one(fs, 1)) else '')
    return uri, _pb_str(fs, 5)

def project_conv(
    cwd='.', # Project directory
    agy_home=None, # Antigravity app data directory, list of directories, or None for CLI and desktop
):
    "The newest top-level `(conv_id, db_path)` for `cwd`"
    target, found, seen = Path(cwd).resolve().as_uri(), {}, set()
    for r in _summary_rows(agy_home):
        seen.add(r.cid)
        if r.parent: continue
        try: uris = json.loads(r.workspace_uris) if r.workspace_uris else []
        except ValueError: uris = []
        if target in uris and (p := _db_for_summary(r)): found.setdefault(r.cid, p)
    for h in agy_homes(agy_home):
        cdir = h/'conversations'
        if not cdir.exists(): continue
        for p in cdir.glob('*.db'):
            if p.stem in seen: continue
            uri, parent = _db_info(p)
            if not parent and uri == target: found[p.stem] = p
    if not found: raise FileNotFoundError(f'No Antigravity conversation for {Path(cwd).resolve()}')
    return max(found.items(), key=lambda it: db_mtime_ns(it[1]))

In [ ]:
#| export
def _conv_by_title(title, agy_home=None):
    for r in _summary_rows(agy_home):
        if r.title == title and (p := _db_for_summary(r)): return r.cid, p
    for h in agy_homes(agy_home):
        adir = h/'annotations'
        if not adir.exists(): continue
        for p in sorted(adir.glob('*.pbtxt'), key=lambda o: o.stat().st_mtime, reverse=True):
            if m := re.search(r'title:\s*"([^"]+)"', p.read_text()):
                if m[1] == title and (db := conv_file(p.stem, agy_home)): return db.stem, db
    return None

def resolve_conv(
    ref=None, # Conversation id, prefix, or title; current or project's newest if None
    cwd='.', # Project directory when resolving the newest conversation
    agy_home=None, # Antigravity app data directory, list of directories, or None for CLI and desktop
):
    "Resolve a conversation reference to `(conv_id, db_path)`"
    if ref is None:
        if (cid := cur_conv()) and (p := conv_file(cid, agy_home)): return cid, p
        return project_conv(cwd, agy_home)
    if p := conv_file(ref, agy_home): return p.stem, p
    if found := _conv_by_title(ref, agy_home): return found
    raise FileNotFoundError(f'No Antigravity conversation for {ref!r}')

## Building synthetic conversations

These test helpers construct valid SQLite conversation databases with real protobuf wire records matching Antigravity's table and field numbers. Keeping fixtures synthetic lets every notebook example run fast without checking binary databases into git.

In [ ]:
#| hide
def _mk_meta(idx, source, traj_id, conv_id, step_id=None, ts=None, call_pb=None):
    t = _pb_ts_bytes(ts)
    sid = step_id or uuid.uuid4().hex
    ref = _pb_f_bytes(1, traj_id) + _pb_f_int(2, idx) + _pb_f_bytes(4, conv_id)
    tr = _pb_f_bytes(1, _pb_f_int(1, STATUS_DONE) + _pb_f_bytes(2, t))
    m = _pb_f_bytes(1, t) + _pb_f_int(3, source)
    if call_pb: m += _pb_f_bytes(4, call_pb)
    return m + _pb_f_bytes(8, t) + _pb_f_bytes(20, ref) + _pb_f_bytes(26, tr) + _pb_f_bytes(32, t) + _pb_f_bytes(35, sid)

def _mk_call_pb(call_id, name, args): return _pb_f_bytes(1, call_id) + _pb_f_bytes(2, name) + _pb_f_bytes(3, json.dumps(args))

def _mk_step(idx, step_type, meta, field_num, body, status=STATUS_DONE, err=None):
    payload = _pb_f_int(1, step_type) + _pb_f_int(4, status) + _pb_f_bytes(5, meta) + _pb_f_bytes(field_num, body)
    return (idx, step_type, status, 0, meta, err, None, None, None, payload, 0)

def _mk_user(idx, text, traj_id, conv_id, ts=None):
    meta = _mk_meta(idx, 4, traj_id, conv_id, ts=ts)
    return _mk_step(idx, STEP_USER, meta, 19, _pb_f_bytes(2, text))

def _mk_asst(idx, text, traj_id, conv_id, calls=(), thinking='', ts=None):
    meta = _mk_meta(idx, 2, traj_id, conv_id, ts=ts)
    body = _pb_f_bytes(1, text) + (_pb_f_bytes(3, thinking) if thinking else b'')
    for cid, name, args in calls: body += _pb_f_bytes(7, _mk_call_pb(cid, name, args))
    return _mk_step(idx, STEP_ASST, meta, 20, body)

def _mk_tool(idx, call_id, name, args, result, traj_id, conv_id, mcp_raw=None, mcp_uri=None, status=STATUS_DONE, err_msg=None, ts=None):
    cpb = _mk_call_pb(call_id, name, args)
    meta = _mk_meta(idx, 2, traj_id, conv_id, ts=ts, call_pb=cpb)
    res_pb = _pb_f_bytes(1, result) if result else b''
    if mcp_raw is not None or mcp_uri is not None:
        inner = _pb_f_bytes(1, args.get('ServerName', ''))
        if mcp_raw is not None: inner += _pb_f_bytes(3, mcp_raw)
        if mcp_uri is not None: inner += _pb_f_bytes(4, mcp_uri)
        any_pb = _pb_f_bytes(1, 'type.googleapis.com/exa.cortex.core.tools.McpTool') + _pb_f_bytes(2, inner)
        res_pb += _pb_f_bytes(6, any_pb)
    err = _pb_f_bytes(2, err_msg) if err_msg else None
    return _mk_step(idx, STEP_TOOL, meta, 140, _pb_f_bytes(2, res_pb), status=status, err=err)

def _mk_sys(idx, text, tag, traj_id, conv_id, ts=None):
    meta = _mk_meta(idx, 5, traj_id, conv_id, ts=ts)
    body = _pb_f_bytes(1, text) + (_pb_f_bytes(3, tag) if tag else b'')
    return _mk_step(idx, STEP_SYS, meta, 114, body)

def _init_conv_db(path, cid, traj_id, cwd, rows, title='', agy_home=None, parent=''):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with sqlite3.connect(path) as con:
        con.executescript('''
            CREATE TABLE trajectory_meta (trajectory_id text PRIMARY KEY, cascade_id text, trajectory_type integer, source integer);
            CREATE TABLE steps (idx integer PRIMARY KEY, step_type integer NOT NULL DEFAULT 0, status integer NOT NULL DEFAULT 0,
                has_subtrajectory numeric NOT NULL DEFAULT false, metadata blob, error_details blob, permissions blob,
                task_details blob, render_info blob, step_payload blob, step_format integer NOT NULL DEFAULT 0);
            CREATE TABLE gen_metadata (idx integer PRIMARY KEY, data blob, size integer NOT NULL DEFAULT 0);
            CREATE TABLE trajectory_metadata_blob (id text PRIMARY KEY DEFAULT "main", data blob);
        ''')
        con.execute('INSERT INTO trajectory_meta VALUES (?, ?, 4, 17)', (traj_id, cid))
        con.executemany('INSERT INTO steps VALUES (?,?,?,?,?,?,?,?,?,?,?)', rows)
        uri = Path(cwd).resolve().as_uri()
        tmb = _pb_f_bytes(7, uri) + (_pb_f_bytes(5, parent) if parent else b'')
        con.execute("INSERT INTO trajectory_metadata_blob VALUES ('main', ?)", (tmb,))
    if agy_home is not None:
        sdb = Path(agy_home)/'conversation_summaries.db'
        with sqlite3.connect(sdb) as con:
            con.execute('''CREATE TABLE IF NOT EXISTS conversation_summaries (
                conversation_id text PRIMARY KEY, title text NOT NULL DEFAULT "", step_count integer NOT NULL DEFAULT 0,
                last_modified_time datetime NOT NULL, workspace_uris text NOT NULL,
                parent_conversation_id text NOT NULL DEFAULT "", app_data_dir text NOT NULL DEFAULT "")''')
            con.execute('INSERT OR REPLACE INTO conversation_summaries VALUES (?,?,?,?,?,?,?)',
                (cid, title, len(rows), '2026-11-20 02:56:12+00:00', json.dumps([uri]), parent, 'antigravity-cli'))
    return Path(path)

Create a temporary Antigravity home with one conversation that uses the persistent kernel (including a `𝍁…𝍁` full-string output), `run_command`, `read_file` on a `SKILL.md`, a hook system message, and an agent report:

In [ ]:
home = Path(tempfile.mkdtemp())
proj = home/'project'
proj.mkdir()
cid, tid = 'ab12cd34-0000-4000-8000-000000000000', '11111111-0000-4000-8000-000000000000'
t0 = datetime(2026, 11, 20, 2, 0, 0, tzinfo=timezone.utc)
t1 = datetime(2026, 11, 20, 2, 1, 0, tzinfo=timezone.utc)
t2 = datetime(2026, 11, 20, 2, 2, 0, tzinfo=timezone.utc)

mcp_args = dict(ServerName='aai-coding_clikernel', ToolName='exec', Arguments=dict(code='6 * 7'))
sample_rows = [_mk_user(0, 'Check the flux meter skill and compute 6*7.', tid, cid, ts=t0),
    _mk_sys(1, 'The following system message was injected by a SessionStart hook:\nnbdev project.', 'hook_json_SessionStart', tid, cid, ts=t0),
    _mk_asst(2, 'Reading the skill and running the kernel.', tid, cid, thinking='Private thought.',
        calls=[('c1', 'read_file', dict(Path='/skills/flux/SKILL.md')),
            ('c2', 'call_mcp_tool', mcp_args),
            ('c3', 'run_command', dict(CommandLine='printf "flux: 41.7\\n"'))], ts=t0),
    _mk_tool(3, 'c1', 'read_file', dict(Path='/skills/flux/SKILL.md'),
        'Created At: 2026-11-20T12:00:00+10:00\nCompleted At: 2026-11-20T12:00:01+10:00\nShowing lines 1-2 of 2\n1: # Flux skill', tid, cid, ts=t0),
    _mk_tool(4, 'c2', 'call_mcp_tool', mcp_args,
        'A pre-tool hook changed the arguments of this tool call before it ran.\n\nCreated At: 2026-11-20T12:00:01+10:00\nCompleted At: 2026-11-20T12:00:02+10:00\n𝍁42𝍁',
        tid, cid, mcp_raw='𝍁42𝍁', ts=t0),
    _mk_tool(5, 'c3', 'run_command', dict(CommandLine='printf "flux: 41.7\\n"'),
        'Created At: 2026-11-20T12:00:02+10:00\nCompleted At: 2026-11-20T12:00:03+10:00\nCommand exited with code 0.\n\nCommand output:\nflux: 41.7', tid, cid, ts=t0),
    _mk_asst(6, 'The kernel returned 42 and the flux reading is 41.7.', tid, cid, ts=t1),
    _mk_sys(7, '[Message] timestamp=2026-11-20T02:02:00Z sender=e6572eae priority=MESSAGE_PRIORITY_HIGH content=Subagent verified flux units.', 'agent_message', tid, cid, ts=t2)]
db_path = _init_conv_db(home/'conversations'/f'{cid}.db', cid, tid, proj, sample_rows, title='Flux Check', agy_home=home)

Resolve the conversation by id prefix, path, project directory, or title; `project_conv` skips child conversations (summarized or not), ignores empty `-wal` files, and reads closed databases without creating `-wal` files:

In [ ]:
test_eq(conv_file('ab12cd34', home), db_path)
test_eq(project_conv(proj, home), (cid, db_path))
test_eq(resolve_conv('Flux Check', proj, home), (cid, db_path))
test_eq(conv_file(db_path), db_path)
w1_cid, w2_cid = 'cd34ef56-1111-4000-8000-000000000000', 'ef56ab78-2222-4000-8000-000000000000'
w1 = _init_conv_db(home/'conversations'/f'{w1_cid}.db', w1_cid, tid, proj, [], agy_home=home, parent=cid)
w2 = _init_conv_db(home/'conversations'/f'{w2_cid}.db', w2_cid, tid, proj, [], parent=cid)
con = sqlite3.connect(w2)
con.execute('PRAGMA journal_mode=WAL')
con.close()
os.utime(w1, ns=(db_path.stat().st_atime_ns, db_path.stat().st_mtime_ns + 2_000_000_000))
wal0 = Path(f'{db_path}-wal')
wal0.write_bytes(b'')
os.utime(wal0, ns=(db_path.stat().st_atime_ns, db_path.stat().st_mtime_ns + 3_000_000_000))
test_eq(db_mtime_ns(db_path), db_path.stat().st_mtime_ns)
test_eq(project_conv(proj, home), (cid, db_path))
assert not Path(f'{w2}-wal').exists()
wal0.unlink()
resolve_conv('ab12cd34', proj, home)

## Reading and searching a conversation

`load_steps` decodes every row in `steps` into a uniform `AttrDict`. For `call_mcp_tool` calls, `_norm_call` unwraps `ServerName`, `ToolName`, and `Arguments` into `mcp__<server>__<tool>` so downstream compaction and dialog tools treat persistent-kernel calls like those from Claude Code and Codex. For MCP results, `_tool_result_text` reads the raw `McpTool` payload (or an offloaded `output.txt` when it carries `𝍁…𝍁` markers) so Antigravity's hook and timestamp prefixes never hide untruncated `FullString` outputs.

In [ ]:
#| export
_re_tool_hdr = re.compile(r'^(?:A pre-tool hook changed the arguments of this tool call before it ran\.[^\n]*\n+)?'
    r'(?:Created At: [^\n]+\n)?(?:Completed At: [^\n]+\n)?')

def _norm_call(name, args):
    "Unwrap `call_mcp_tool` into `mcp__<server>__<tool>` with its inner arguments"
    if name == 'call_mcp_tool' and isinstance(args, dict) and 'ServerName' in args and 'ToolName' in args:
        inner = args.get('Arguments')
        return f"mcp__{args['ServerName']}__{args['ToolName']}", inner if isinstance(inner, dict) else {}
    return name, args if isinstance(args, dict) else {}

def _uri_path(uri):
    if not uri or not uri.startswith('file://'): return None
    return Path(unquote(urlparse(uri).path))

def _read_full_uri(uri):
    p = _uri_path(uri)
    if not p or not p.exists(): return None
    s = p.read_text(errors='replace').strip()
    return s if len(s) > 2 and s[0] == '𝍁' and s[-1] == '𝍁' else None

def _tool_result_text(f140, err_bytes, status):
    "Extract clean tool result text, preserving raw MCP `𝍁…𝍁` strings and error details"
    f2 = _pb_decode(_pb_one(f140, 2, b'')) if f140 else []
    for any_b in _pb_all(f2, 6):
        af = _pb_decode(any_b)
        if _pb_str(af, 1).endswith('.McpTool'):
            mf = _pb_decode(_pb_one(af, 2, b''))
            if raw := _pb_str(mf, 3): return raw
            if full_txt := _read_full_uri(_pb_str(mf, 4)): return full_txt
    txt = _re_tool_hdr.sub('', _pb_str(f2, 1)).strip()
    if txt: return txt
    if err_bytes:
        ef = _pb_decode(err_bytes)
        if msg := (_pb_str(ef, 2) or _pb_str(ef, 3)): return msg
    return 'Canceled' if status == STATUS_CANCELED else ''

In [ ]:
#| export
def _parse_call(buf):
    fs = _pb_decode(buf)
    raw_args = _pb_str(fs, 3)
    try: args = json.loads(raw_args) if raw_args else {}
    except ValueError: args = dict(raw=raw_args)
    name, args = _norm_call(_pb_str(fs, 2), args)
    return AttrDict(id=_pb_str(fs, 1), name=name, arguments=args)

def _parse_step(idx, step_type, status, meta_b, err_b, payload_b):
    pf = _pb_decode(payload_b)
    mf = _pb_decode(meta_b or _pb_one(pf, 5, b''))
    base = AttrDict(idx=idx, step_type=step_type, status=status, created=_pb_ts(_pb_one(mf, 1)), step_id=_pb_str(mf, 35))
    if step_type == STEP_USER:
        f19 = _pb_decode(_pb_one(pf, 19, b''))
        txt = _pb_str(f19, 2) or ''.join(_pb_str(_pb_decode(b), 1) for b in _pb_all(f19, 3))
        return AttrDict(**base, role='user', text=txt)
    if step_type == STEP_ASST:
        f20 = _pb_decode(_pb_one(pf, 20, b''))
        calls = [_parse_call(b) for b in _pb_all(f20, 7)]
        return AttrDict(**base, role='assistant', text=_pb_str(f20, 1), thinking=_pb_str(f20, 3), tool_calls=calls)
    if step_type == STEP_TOOL:
        cb = _pb_one(mf, 4)
        call = _parse_call(cb) if cb else AttrDict(id='', name='', arguments={})
        f140 = _pb_decode(_pb_one(pf, 140, b''))
        txt = _tool_result_text(f140, err_b, status)
        full = len(txt) > 2 and txt[0] == '𝍁' and txt[-1] == '𝍁'
        return AttrDict(**base, role='tool', call_id=call.id, tool_name=call.name, tool_args=call.arguments, text=txt, full=full)
    if step_type == STEP_SYS:
        f114 = _pb_decode(_pb_one(pf, 114, b''))
        return AttrDict(**base, role='system', text=_pb_str(f114, 1), tag=_pb_str(f114, 3))
    if step_type == STEP_CHECKPOINT:
        f30 = _pb_decode(_pb_one(pf, 30, b''))
        return AttrDict(**base, role='checkpoint', text=_pb_str(f30, 5), cutoff=_pb_one(f30, 12, idx), checkpoint_id=_pb_str(f30, 29))
    return AttrDict(**base, role='other', text='')

def load_steps(
    path, # Path to a conversation SQLite `.db` file
):
    "Decode every step in `path` in index order"
    with _ro_conn(path) as con:
        rows = con.execute('SELECT idx, step_type, status, metadata, error_details, step_payload FROM steps ORDER BY idx').fetchall()
    return L(_parse_step(*r) for r in rows)

def load_conv(
    ref=None, # Conversation id, prefix, or title; current or project's newest if None
    cwd='.', # Project directory
    agy_home=None, # Antigravity app data directory, list of directories, or None for CLI and desktop
):
    "Load all decoded steps for conversation `ref`"
    _, path = resolve_conv(ref, cwd, agy_home)
    return load_steps(path)

Load the synthetic conversation and verify that `call_mcp_tool` unwrapped cleanly and the `𝍁42𝍁` result kept its `full` flag:

In [ ]:
steps = load_steps(db_path)
test_eq(Counter(steps.attrgot('role')), Counter(user=1, system=2, assistant=2, tool=3))
test_eq(steps[2].tool_calls[1].name, 'mcp__aai-coding_clikernel__exec')
test_eq((steps[4].tool_name, steps[4].text, steps[4].full), ('mcp__aai-coding_clikernel__exec', '𝍁42𝍁', True))
[(s.idx, s.role, s.get('tool_name', s.get('tag', ''))) for s in steps]

`step_search` and `show_steps` mirror `ant.sess_search`/`show_recs` and `oai.item_search`/`show_items`. Each search hit keeps its decoded step on `.step` and the full searched list on `.steps`.

In [ ]:
#| export
def _trunc_sz(s, mx): return truncstr(s, mx, suf=f'…[{humanize(len(s))}]')

def _step_txt(s):
    if s.role == 'assistant':
        calls = [f"{c.name}({json.dumps(c.arguments)})" for c in s.get('tool_calls', ())]
        return '\n'.join(filter(None, [s.text, *calls]))
    return s.get('text', '')

class StepHits(L):
    "Search hits with one match-centered preview per line, each carrying its step on `.step`"
    steps = None
    def _new(self, items, *args, **kw):
        res = super()._new(items, *args, **kw)
        res.steps = self.steps
        return res
    def __repr__(self): return '\n'.join(f"{h.idx:5} {h.role:10} {h.preview}" for h in self)

def step_search(
    pat, # Regex to find
    steps, # Decoded steps, e.g. from `load_steps`
    maxlen=MAXLEN, # Preview characters around the match
):
    "Search decoded steps: `StepHits` rows, each carrying its step on `.step` and all searched steps on `.steps`"
    steps, res = L(steps), StepHits()
    for i, s in enumerate(steps):
        text = _step_txt(s)
        if m := re.search(pat, text):
            h = maxlen // 2
            a, b = max(0, m.start()-h), min(len(text), m.end()+h)
            prev = ('…' if a else '') + text[a:b].replace('\n', ' ') + ('…' if b < len(text) else '')
            if a or b < len(text): prev += f'[{humanize(len(text))}]'
            res.append(AttrDict(i=i, idx=s.idx, role=s.role, preview=prev, step=s))
    res.steps = steps
    return res

def show_steps(
    steps, # Decoded steps, e.g. a slice of `StepHits.steps`
    mx=500, # Text characters per step
):
    "Readable view of decoded steps"
    rows = [f"--- #{s.idx} {s.role} ---\n{_trunc_sz(_step_txt(s), mx)}" for s in steps]
    return PrettyString('\n'.join(rows))

In [ ]:
hits = step_search(r'41\.7', steps)
test_eq([h.role for h in hits], ['assistant', 'tool', 'assistant'])
test_eq(type(hits[:1]).__name__, 'StepHits')
show_steps(hits.steps[hits[1].i-1:hits[1].i+2], mx=120)

## Prompt history

`prompt_hist` recovers user prompts from Antigravity conversation databases across CLI and desktop stores. Each row identifies the timestamp, project directory, conversation id, and prompt text.

In [ ]:
#| export
class PromptHist(list):
    "Antigravity prompt-history rows, one line per user message"
    def __repr__(self):
        return '\n'.join(f"{h.ts[:16]} {Path(h.project).name:>12} {h.conv[:8]} {_trunc_sz(h.text.replace(chr(10),' '), 100)}" for h in self)

def prompt_hist(
    project=None, # Only conversations whose workspace is this path
    since=None, # Only timestamps at or after this ISO string
    pat=None, # Only prompt text matching this regex
    agy_home=None, # Antigravity app data directory, list of directories, or None for CLI and desktop
):
    "User prompts retained in Antigravity conversation databases, oldest first"
    proj_uri = Path(project).expanduser().resolve().as_uri() if project else None
    rows = []
    for h in agy_homes(agy_home):
        cdir = h/'conversations'
        if not cdir.exists(): continue
        for path in sorted(cdir.glob('*.db')):
            ws_uri, parent = _db_info(path)
            if parent or (proj_uri and ws_uri != proj_uri): continue
            cwd = str(_uri_path(ws_uri)) if ws_uri else ''
            for s in load_steps(path):
                if s.step_type != STEP_USER or s.text.strip() in ('', '/compact'): continue
                if since and s.created < since: continue
                if pat and not re.search(pat, s.text): continue
                rows.append(AttrDict(ts=s.created, conv=path.stem, project=cwd, text=s.text, path=path))
    return PromptHist(sorted(rows, key=lambda x: x.ts))

In [ ]:
ph = prompt_hist(proj, pat='flux', agy_home=home)
test_eq(len(ph), 1)
test_eq(ph[0].conv, cid)
ph

## Active history and dialogs

`split_compaction` locates the latest `CHECKPOINT` step (`step_type = 23`). For an `llmsurgery` synthetic checkpoint (whose text contains `\n\n## Conversation\n\n`), it extracts the unchanged DSL body with `compact_body`. For a native `/compact` checkpoint, it keeps the native summary text. `active_steps` returns steps from that latest checkpoint onward.

In [ ]:
#| export
def split_compaction(
    steps, # Decoded steps from `load_steps`
):
    "Return `(prior_summary, new_steps)` split at the latest `CHECKPOINT` step"
    for i in range(len(steps)-1, -1, -1):
        if steps[i].step_type == STEP_CHECKPOINT:
            raw = steps[i].text
            prior = compact_body(raw) if '\n\n## Conversation\n\n' in raw else raw.strip()
            return prior, L(steps[i+1:])
    return '', L(steps)

def active_steps(
    steps, # Decoded steps from `load_steps`
):
    "Steps from the latest `CHECKPOINT` onward, or all steps if uncompacted"
    for i in range(len(steps)-1, -1, -1):
        if steps[i].step_type == STEP_CHECKPOINT: return L(steps[i:])
    return L(steps)

`steps2chat` converts decoded steps into canonical `Msg` objects with `created` and `uid` metadata. It drops hook plumbing (`hook_json_*`), fork notices, and `sender=system` restart notices. With `agent_results=True`, it retains `agent_message`, `task_notification`, `plan_mode_handoff`, and untagged `[Message]` steps so compaction preserves subagent reports under their bounded budget.

In [ ]:
#| export
_skip_sys_tags = frozenset(('hook_json_PreInvocation', 'hook_json_SessionStart', 'fork_notice'))

def _is_agent_sys(s):
    if s.tag in _skip_sys_tags: return False
    if 'sender=system ' in s.text: return False
    return s.tag in ('agent_message', 'task_notification', 'plan_mode_handoff') or s.text.startswith('[Message] ')

def steps2chat(
    steps, # Decoded steps from `load_steps`
    agent_results=False, # Include subagent messages, task notifications, and plan handoffs as user turns?
):
    "Convert Antigravity steps into canonical `Msg` objects"
    msgs = []
    def add(m, s):
        m.meta = dict(created=s.created, uid=s.step_id)
        msgs.append(m)
    for s in steps:
        if s.step_type == STEP_USER:
            if s.text.strip() in ('', '/compact'): continue
            add(Msg('user', [Text(s.text)]), s)
        elif s.step_type == STEP_ASST:
            parts = ([Text(s.text)] if s.text else []) + [ToolUse(c.id, c.name, c.arguments) for c in s.tool_calls]
            if parts: add(Msg('assistant', parts), s)
        elif s.step_type == STEP_TOOL:
            add(Msg('tool', [ToolResult(id=s.call_id, name=s.tool_name, arguments=s.tool_args, text=s.text, full=s.full)]), s)
        elif s.step_type == STEP_SYS and _is_agent_sys(s):
            if agent_results: add(Msg('user', [Text(s.text)]), s)
            elif not msgs:
                body = s.text.split(' content=', 1)[1].strip() if s.text.startswith('[Message] ') and ' content=' in s.text else s.text
                if body: add(Msg('user', [Text(body)]), s)
    return msgs

def conv2dlg(
    ref=None, # Conversation id, prefix, or title; current or project's newest if None
    cwd='.', # Project directory
    agy_home=None, # Antigravity app data directory, list of directories, or None for CLI and desktop
    name=None, # Dialog name; conversation id if None
    mx=2000, # Maximum rendered tool string length
    since_compact=False, # Only the conversation since the last checkpoint?
    steps=None, # Optional preloaded steps
):
    "Convert an Antigravity conversation into an editable aidialog `Dialog`"
    if steps is None:
        cid, path = resolve_conv(ref, cwd, agy_home)
        steps = load_steps(path)
        name = name or cid
    if since_compact: steps = active_steps(steps)
    return chat2dlg(steps2chat(steps), name or 'conversation', mx=mx)

In [ ]:
dlg = conv2dlg('ab12cd34', proj, home)
test_eq(len(dlg.messages), 1)
test('41.7', dlg.messages[0].ai_res, in_)
dlg.summary()

Subagent conversations start with an `agent_message` system step instead of a `USER_INPUT` step. `steps2chat` turns that opening message into the dialog's first user prompt:

In [ ]:
def _row2step(r): return _parse_step(r[0], r[1], r[2], r[4], r[5], r[9])
sub_msgs = steps2chat([
    _row2step(_mk_sys(0, '[Message] timestamp=2026-11-20T02:00:00Z sender=parent priority=MESSAGE_PRIORITY_HIGH content=Check the tests.', 'agent_message', tid, cid, ts=t0)),
    _row2step(_mk_asst(1, 'All tests pass.', tid, cid, ts=t1))])
test_eq([m.role for m in sub_msgs], ['user', 'assistant'])
test_eq(sub_msgs[0].content[0].text, 'Check the tests.')
sub_msgs[0].content[0].text

## Synthetic compaction

Antigravity compacts a conversation by appending one `CHECKPOINT` row (`step_type = 23`, `status = 3`) to `steps` without rewriting prior rows or `gen_metadata`. When resumed, Antigravity builds `<CONTEXT_SUMMARY>` from payload field `30 -> 5`, stops before the cutoff index in `30 -> 12`, and uses `30 -> 29` as the rendered summary message's identifier.

`transcript_path` points the continuation document to `brain/<id>/.system_generated/logs/transcript_full.jsonl` when present, falling back to the `.db` path.

In [ ]:
#| export
def checkpoint_step(
    idx, # Step index for the new `CHECKPOINT` row
    content, # Continuation document stored in checkpoint field `5`
    traj_id, # Owning trajectory UUID from `trajectory_meta`
    conv_id, # Conversation (cascade) UUID
    gen_count=0, # Current `gen_metadata` row count
    ts=None, # Timestamp datetime; current UTC time if None
):
    "Build protobuf `metadata` and `step_payload` bytes for a `CHECKPOINT` (`step_type = 23`) step"
    t = _pb_ts_bytes(ts)
    step_id, cid_a, cid_b = uuid.uuid4().hex, uuid.uuid4().hex, uuid.uuid4().hex
    tref = _pb_f_bytes(1, traj_id) + _pb_f_int(2, idx) + (_pb_f_int(3, gen_count) if gen_count else b'') + _pb_f_bytes(4, conv_id)
    trans = _pb_f_bytes(1, _pb_f_int(1, STATUS_DONE) + _pb_f_bytes(2, t))
    meta = (_pb_f_bytes(1, t) + _pb_f_int(3, 5) + _pb_f_bytes(8, t) + _pb_f_bytes(12, str(uuid.uuid4()))
        + _pb_f_bytes(20, tref) + _pb_f_bytes(26, trans) + _pb_f_bytes(32, t) + _pb_f_bytes(35, step_id))
    ckpt = (_pb_f_bytes(5, content) + _pb_f_int(12, idx) + _pb_f_int(24, 1)
        + _pb_f_int(27, 0) + _pb_f_bytes(28, cid_a) + _pb_f_bytes(29, cid_b))
    payload = _pb_f_int(1, STEP_CHECKPOINT) + _pb_f_int(4, STATUS_DONE) + _pb_f_bytes(5, meta) + _pb_f_bytes(30, ckpt)
    return AttrDict(idx=idx, step_type=STEP_CHECKPOINT, status=STATUS_DONE,
        metadata=meta, step_payload=payload, step_id=step_id, checkpoint_id=cid_b)

def transcript_path(
    conv_id, # Conversation UUID
    db_path, # Path to `<conv_id>.db`
):
    "Path to `transcript_full.jsonl` when present, otherwise `db_path`"
    p = Path(db_path).parents[1]/'brain'/conv_id/'.system_generated'/'logs'/'transcript_full.jsonl'
    return p if p.exists() else Path(db_path)

In [ ]:
#| export
def prepare_compaction(
    ref=None, # Conversation id, prefix, or title; current or project's newest if None
    cwd='.', # Project directory
    agy_home=None, # Antigravity app data directory, list of directories, or None for CLI and desktop
    policy=compact_policy, # Token budget dictionary for `compact_chat`
    enc=None, # Tokenizer encoding; `compact_enc()` if None
):
    "Prepare an incremental synthetic compaction without writing to SQLite"
    cid, path = resolve_conv(ref, cwd, agy_home)
    steps = load_steps(path)
    with _ro_conn(path) as con:
        row = con.execute('SELECT trajectory_id FROM trajectory_meta LIMIT 1').fetchone()
        gen_count = con.execute('SELECT count(*) FROM gen_metadata').fetchone()[0]
    traj_id = row[0] if row else cid
    prior, new_steps = split_compaction(steps)
    msgs = steps2chat(new_steps, agent_results=True)
    if not msgs and not prior: raise ValueError(f'No steps to compact in {cid}')
    new_chat = compact_chat(msgs, enc=enc, last_n=5, **policy)
    new_full = compact_chat(msgs, enc=enc, **{k:10**9 for k in policy})
    chat, full = join_compacts(prior, new_chat), join_compacts(prior, new_full)
    content = compact_content(chat, transcript_path(cid, path))
    next_idx = (max(s.idx for s in steps) + 1) if steps else 0
    step = checkpoint_step(next_idx, content, traj_id, cid, gen_count)
    return AttrDict(cid=cid, path=path, step=step, prior=prior, new_chat=new_chat, chat=chat, full=full,
        content=content, pre_toks=len_toks(full, enc), post_toks=len_toks(content, enc))

def append_compaction(
    compaction, # Result of `prepare_compaction`
):
    "Insert a prepared `CHECKPOINT` row into the conversation database and update `conversation_summaries.db`"
    s = compaction.step
    sql = ('INSERT INTO steps (idx, step_type, status, has_subtrajectory, metadata, step_payload, step_format) '
        'VALUES (?, ?, ?, 0, ?, ?, 0)')
    with sqlite3.connect(compaction.path) as con: con.execute(sql, (s.idx, s.step_type, s.status, s.metadata, s.step_payload))
    sdb = Path(compaction.path).parents[1]/'conversation_summaries.db'
    if sdb.exists():
        now = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S.%f+00:00')
        with sqlite3.connect(sdb) as con:
            con.execute('UPDATE conversation_summaries SET step_count=?, last_modified_time=? WHERE conversation_id=?',
                (s.idx + 1, now, compaction.cid))
    return compaction.path

def compact_session(
    ref=None, # Conversation id, prefix, or title; current or project's newest if None
    cwd='.', # Project directory
    agy_home=None, # Antigravity app data directory, list of directories, or None for CLI and desktop
    policy=compact_policy, # Token budget dictionary for `compact_chat`
    enc=None, # Tokenizer encoding; `compact_enc()` if None
):
    "Generate and append a synthetic `CHECKPOINT` compaction"
    compaction = prepare_compaction(ref, cwd, agy_home, policy, enc)
    append_compaction(compaction)
    return compaction

Prepare a compaction first and inspect it before writing. The `clikernel` call renders as fenced Python, `𝍁42𝍁` stays marked untruncated, `run_command` renders with `!`, the agent message uses `> Agent:`, and the footer reminds the resumed session to re-read `/skills/flux/SKILL.md`:

In [ ]:
before_n = len(load_steps(db_path))
comp = prepare_compaction('ab12cd34', proj, home)
test_eq(len(load_steps(db_path)), before_n)
test('```python\n6 * 7\n```', comp.chat, in_)
test('𝍁> 42𝍁', comp.chat, in_)
test('! printf "flux: 41.7\\n"', comp.chat, in_)
test('> Agent: Subagent verified flux units.', comp.chat, in_)
test('re-read: /skills/flux/SKILL.md', comp.content, in_)
comp.pre_toks, comp.post_toks, comp.step.idx

Append the checkpoint and confirm it becomes the latest step in the database:

In [ ]:
append_compaction(comp)
after = load_steps(db_path)
test_eq(len(after), before_n + 1)
test_eq(after[-1].step_type, STEP_CHECKPOINT)
test_eq(split_compaction(after), (comp.chat, L()))
test_eq(_summary_rows(home)[0].cid, cid)
after[-1].step_type, after[-1].idx

Append a second exchange and compact again with `compact_session`. The new checkpoint keeps the earlier DSL body unchanged and appends only the new turns:

In [ ]:
more = [_mk_user(9, 'Now verify the units.', tid, cid, ts=t2),
    _mk_asst(10, 'Units confirmed: kilofinches.', tid, cid, ts=t2)]
with sqlite3.connect(db_path) as con: con.executemany('INSERT INTO steps VALUES (?,?,?,?,?,?,?,?,?,?,?)', more)
comp2 = compact_session(cid, proj, home)
test_eq(comp2.prior, comp.chat)
test('Now verify the units.', comp2.new_chat, in_)
test('Units confirmed: kilofinches.', comp2.chat, in_)
test_eq(load_steps(db_path)[-1].idx, 11)
comp2.chat

A native Antigravity `/compact` checkpoint holds a `<summary>...</summary>` block instead of `## Conversation`. Subsequent `compact_session` calls keep that native summary verbatim and compact only the steps added after it:

In [ ]:
nat_cid = 'cd34ef56-0000-4000-8000-000000000000'
nat_ckpt = checkpoint_step(1, '<summary>\nNative summary of earlier work.\n</summary>', tid, nat_cid)
nat_rows = [_mk_user(0, 'Old turn before native compact.', tid, nat_cid, ts=t0),
    (1, STEP_CHECKPOINT, STATUS_DONE, 0, nat_ckpt.metadata, None, None, None, None, nat_ckpt.step_payload, 0),
    _mk_user(2, 'Continue after native compact.', tid, nat_cid, ts=t1),
    _mk_asst(3, 'Continuing now.', tid, nat_cid, ts=t1)]
nat_db = _init_conv_db(home/'conversations'/f'{nat_cid}.db', nat_cid, tid, proj, nat_rows, agy_home=home)
nat_comp = prepare_compaction(nat_cid, proj, home)
test_eq(nat_comp.prior, '<summary>\nNative summary of earlier work.\n</summary>')
assert 'Old turn before native compact.' not in nat_comp.chat
test('Continue after native compact.', nat_comp.chat, in_)
nat_comp.chat

## Cleanup

Remove the temporary Antigravity home directory.

In [ ]:
shutil.rmtree(home)

## Export -

In [ ]:
#| hide
#| eval: false
import nbdev; nbdev.nbdev_export()